In [10]:
import re
from collections import Counter
import pandas as pd
import json
from typing import List, Tuple, Dict, Optional
from warnings import filterwarnings
# Silence some expected warnings
filterwarnings("ignore")

In [11]:
molecules = pd.read_csv(r"E:\Projects\Mycobacterium tuberculosis\03-Machine Learning\input\Valid_12756.csv")

In [12]:
molecules.head(1)

,Molecule ChEMBL ID,MIC,Units,Smiles,Label
0,CHEMBL1086385,0.12,ug.mL-1,CCCCCCCC[C@@H](C)C(=O)N1CCC[C@H]1C(=O)N[C@@H](...,1


In [13]:
# 1) SMILES tokenizer
# Adds "@@" (chirality) and keeps %nn ring indices.
smiles_token_pattern = (
    r"(\[[^\]]+]"           # bracket atoms/groups
    r"|Br?|Cl?"             # halogens
    r"|N|O|S|P|F|I|b|c|n|o|s|p"
    r"|\(|\)|\."            # punctuation
    r"|=|#|-|\+|\\|/"       # bonds/slashes (note: backslash escaped)
    r"|:|~|@@|@"            # chirality: @@ before @
    r"|\?|>>?"              # '>' or '>>' (reaction SMILES)
    r"|\*|\$"
    r"|\%[0-9]{2}"          # two-digit ring index like %10
    r"|[0-9])"              # single-digit ring indices
)
tokenizer_re = re.compile(smiles_token_pattern)

def tokenize(smiles: str) -> List[str]:
    """Tokenize a SMILES string into a list of tokens."""
    if smiles is None:
        return []
    s = str(smiles).strip()
    if not s:
        return []
    return tokenizer_re.findall(s)

In [14]:
# 2) Build vocab from dataset
def build_vocab(smiles_list, min_freq=1, pad_token="<PAD>", unk_token="<UNK>", bos_token="<BOS>", eos_token="<EOS>"):
    tokenized = [tokenize(s) for s in smiles_list]
    # count the number of occurrences of each token in all molecules.
    counter = Counter(token for seq in tokenized for token in seq) 
    # keep tokens above min_freq
    tokens = [t for t,c in counter.items() if c >= min_freq]
    # reserve indices
    idx = 0
    vocab = {}
    vocab[pad_token] = idx; idx += 1
    vocab[unk_token] = idx; idx += 1
    vocab[bos_token] = idx; idx += 1
    vocab[eos_token] = idx; idx += 1
    for t in sorted(tokens):  # sort for reproducibility
        if t in vocab:
            continue
        vocab[t] = idx; idx += 1
    return vocab

In [15]:
smiles_list = molecules['Smiles'].to_list()
vocab = build_vocab(smiles_list)

In [16]:
vocab

{'<PAD>': 0,
 '<UNK>': 1,
 '<BOS>': 2,
 '<EOS>': 3,
 '#': 4,
 '%10': 5,
 '%11': 6,
 '%12': 7,
 '%13': 8,
 '%14': 9,
 '%15': 10,
 '%16': 11,
 '%17': 12,
 '%18': 13,
 '%19': 14,
 '%20': 15,
 '%21': 16,
 '%22': 17,
 '%23': 18,
 '(': 19,
 ')': 20,
 '-': 21,
 '.': 22,
 '/': 23,
 '1': 24,
 '2': 25,
 '3': 26,
 '4': 27,
 '5': 28,
 '6': 29,
 '7': 30,
 '8': 31,
 '9': 32,
 '=': 33,
 'B': 34,
 'Br': 35,
 'C': 36,
 'Cl': 37,
 'F': 38,
 'I': 39,
 'N': 40,
 'O': 41,
 'P': 42,
 'S': 43,
 '[Br-]': 44,
 '[C-]': 45,
 '[C@@H]': 46,
 '[C@@]': 47,
 '[C@H]': 48,
 '[C@]': 49,
 '[Cl-]': 50,
 '[I-]': 51,
 '[K+]': 52,
 '[Li+]': 53,
 '[N+]': 54,
 '[N-]': 55,
 '[N@@]': 56,
 '[N@]': 57,
 '[Na+]': 58,
 '[O-]': 59,
 '[P+]': 60,
 '[S+]': 61,
 '[S-]': 62,
 '[S@+]': 63,
 '[S@@+]': 64,
 '[Se]': 65,
 '[Si]': 66,
 '[n+]': 67,
 '[n-]': 68,
 '[nH]': 69,
 '[se]': 70,
 '\\': 71,
 'c': 72,
 'n': 73,
 'o': 74,
 's': 75}

In [17]:
len(vocab)

76

In [18]:
# 3) save vocab as json format
with open(r'E:\Projects\Mycobacterium tuberculosis\03-Machine Learning\vocab.json', 'w', encoding='utf-8') as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)
# save vocab as txt format
with open(r'E:\Projects\Mycobacterium tuberculosis\03-Machine Learning\vocab.txt', 'w', encoding='utf-8') as f:
    for token, idx in sorted(vocab.items(), key=lambda x: x[1]):
        f.write(f"{token}\t{idx}\n")